In [6]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [4]:

import time, math
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DATA_PATH = "./cache_moses/moses_splits_vocab.pt"
obj = torch.load(DATA_PATH)

train_sm, val_sm, test_sm = obj["splits"]
stoi, itos = obj["stoi"], obj["itos"]
cfg0 = obj.get("cfg", {})

PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"
pad_id = stoi[PAD]
bos_id = stoi[BOS]
eos_id = stoi[EOS]

MAX_LEN = int(cfg0.get("MAX_LEN", 140))  
print("Loaded splits:", len(train_sm), len(val_sm), len(test_sm))
print("Vocab size:", len(itos))
print("MAX_LEN:", MAX_LEN)


def encode(sm: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    ids = [stoi[BOS]] + [stoi[c] for c in sm if c in stoi] + [stoi[EOS]]
    if len(ids) < max_len:
        ids = ids + [stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = stoi[EOS]
    return ids

def decode(ids: List[int], itos: List[str]) -> str:
    out = []
    for i in ids:
        if i == eos_id:
            break
        if i in (pad_id, bos_id):
            continue
        out.append(itos[i])
    return "".join(out)

class SmilesLMDataset(Dataset):
    def __init__(self, smiles_list: List[str], stoi: Dict[str,int], max_len: int):
        self.smiles = smiles_list
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode(self.smiles[idx], self.stoi, self.max_len)
        x = torch.tensor(ids[:-1], dtype=torch.long)  # (T-1)
        y = torch.tensor(ids[1:], dtype=torch.long)   # (T-1)
        return x, y

train_ds = SmilesLMDataset(train_sm, stoi, MAX_LEN)
val_ds   = SmilesLMDataset(val_sm,   stoi, MAX_LEN)

BATCH_SIZE = 256 if DEVICE == "cuda" else 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(DEVICE=="cuda"), drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=(DEVICE=="cuda"))

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))


@dataclass
class LMConfig:
    vocab_size: int
    emb_dim: int = 256
    hidden_dim: int = 512
    num_layers: int = 2
    dropout: float = 0.1

class GRULM(nn.Module):
    def __init__(self, cfg: LMConfig, pad_id: int):
        super().__init__()
        self.cfg = cfg
        self.pad_id = pad_id
        self.embed = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            input_size=cfg.emb_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(cfg.hidden_dim, cfg.vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T)
        emb = self.embed(x)          # (B, T, E)
        out, h = self.gru(emb, h0)   # out: (B, T, H)
        logits = self.fc(out)        # (B, T, V)
        return logits, h

lm_cfg = LMConfig(vocab_size=len(itos))
model = GRULM(lm_cfg, pad_id).to(DEVICE)
print(model)


def compute_loss(logits, targets, pad_id: int):
    B, T, V = logits.shape
    return F.cross_entropy(
        logits.view(B*T, V),
        targets.view(B*T),
        ignore_index=pad_id
    )

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)

USE_AMP = (DEVICE == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    n = 0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits, _ = model(x)
            loss = compute_loss(logits, y, pad_id)

        if train:
            scaler.scale(loss).backward()
            # unscale before clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

        total_loss += float(loss.item())
        n += 1

    return total_loss / max(1, n)


EPOCHS = 10  
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    t1 = time.time()
    print(f"Epoch {epoch:02d} | train loss {tr:.4f} | val loss {va:.4f} | {(t1-t0):.1f}s")


DEVICE: cuda
Loaded splits: 200000 20000 20000
Vocab size: 29
MAX_LEN: 140
Train batches: 781 Val batches: 79
GRULM(
  (embed): Embedding(29, 256, padding_idx=0)
  (gru): GRU(256, 512, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=512, out_features=29, bias=True)
)
Epoch 01 | train loss 0.7126 | val loss 0.6032 | 14.7s
Epoch 02 | train loss 0.5919 | val loss 0.5779 | 14.7s
Epoch 03 | train loss 0.5739 | val loss 0.5680 | 14.8s
Epoch 04 | train loss 0.5659 | val loss 0.5630 | 14.7s
Epoch 05 | train loss 0.5617 | val loss 0.5613 | 14.8s
Epoch 06 | train loss 0.5590 | val loss 0.5588 | 14.8s
Epoch 07 | train loss 0.5579 | val loss 0.5601 | 14.8s
Epoch 08 | train loss 0.5577 | val loss 0.5584 | 14.8s
Epoch 09 | train loss 0.5570 | val loss 0.5586 | 14.8s
Epoch 10 | train loss 0.5574 | val loss 0.5610 | 14.8s


In [7]:
@torch.no_grad()
def sample_lm_batched(model: GRULM, n: int, max_len: int, temperature: float = 1.0, batch_size: int = 512) -> List[str]:
    model.eval()
    V = model.cfg.vocab_size
    samples = []

    remaining = n
    while remaining > 0:
        B = min(batch_size, remaining)
        remaining -= B

        x = torch.full((B, 1), bos_id, dtype=torch.long, device=DEVICE)  # (B,1)
        h = None
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)

        out_ids = [[] for _ in range(B)]

        for _t in range(max_len - 1):
            logits, h = model(x, h)                 # logits: (B,1,V)
            logits = logits[:, -1, :] / max(1e-8, temperature)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # (B,1)
            nid = next_id.squeeze(1)                            # (B,)

            for i in range(B):
                if finished[i]:
                    continue
                tok = int(nid[i].item())
                if tok == eos_id:
                    finished[i] = True
                elif tok not in (pad_id, bos_id):
                    out_ids[i].append(tok)

            x = next_id
            if finished.all():
                break

        for i in range(B):
            samples.append("".join(itos[t] for t in out_ids[i]))

    return samples


def to_mol(smiles: str):
    return Chem.MolFromSmiles(smiles)

def canonicalize(smiles: str) -> str | None:
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m, canonical=True)

# canonical train set for novelty
train_canon: Set[str] = set()
for s in train_sm:
    cs = canonicalize(s)
    if cs is not None:
        train_canon.add(cs)

def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    valid_canon = []
    for s in smiles_list:
        cs = canonicalize(s)
        if cs is not None:
            valid_canon.append(cs)
    return len(valid_canon) / max(1, len(smiles_list)), valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    return 0.0 if len(valid_canon) == 0 else len(set(valid_canon)) / len(valid_canon)

def novelty_per_sample(valid_canon: List[str], train_set: Set[str]) -> float:
    return 0.0 if len(valid_canon)==0 else sum(1 for s in valid_canon if s not in train_set)/len(valid_canon)

def novelty_unique(valid_canon: List[str], train_set: Set[str]) -> float:
    uv = set(valid_canon)
    return 0.0 if len(uv)==0 else sum(1 for s in uv if s not in train_set)/len(uv)

def evaluate_smiles(samples: List[str], train_set: Set[str]) -> Dict[str, float]:
    v, valid_canon = validity(samples)
    out = {
        "n_samples": len(samples),
        "validity": v,
        "n_valid": len(valid_canon),
        "uniqueness": uniqueness(valid_canon),
        "novelty": novelty_per_sample(valid_canon, train_set),
        "novelty_unique": novelty_unique(valid_canon, train_set),
    }
    out.update(compute_properties(valid_canon))
    return out

N_SAMPLES = 5000
TEMP = 0.9

t0 = time.time()
gen = sample_lm_batched(model, n=5000, max_len=MAX_LEN, temperature=0.9, batch_size=512)
t1 = time.time()
print("sec:", t1-t0, "samples/sec:", 5000/(t1-t0))

metrics = evaluate_smiles(gen, train_canon)
gen_seconds = t1 - t0
metrics["gen_seconds"] = gen_seconds
metrics["samples_per_sec"] = N_SAMPLES / gen_seconds
metrics["valid_per_sec"] = metrics["n_valid"] / gen_seconds

print(metrics)

sec: 4.566656827926636 samples/sec: 1094.8928698612353
{'n_samples': 5000, 'validity': 0.9352, 'n_valid': 4676, 'uniqueness': 0.9997861420017109, 'novelty': 0.9863130881094953, 'novelty_unique': 0.9863101604278075, 'mw_mean': 305.79570744225714, 'logp_mean': 2.4958630367835775, 'qed_mean': 0.8050267869384191, 'gen_seconds': 4.566656827926636, 'samples_per_sec': 1094.8928698612353, 'valid_per_sec': 1023.9438118942273}
